# 장갑 보정 키보드 — 본 실험 분석

이 노트북은 **원자료(참가자 로그)에서 출발해 지표 계산 → 통계 검정 → 그래프까지** 모든 분석을 순서대로 실행한다.
셀을 위에서부터 차례로 실행하면 된다(Shift + Enter). 각 단계의 중간 결과 표가 함께 표시된다.

| 단계 | 내용 | 사용하는 코드 |
|---|---|---|
| 1 | 원자료 불러오기 | `parse_logs.py` |
| 2 | 지표 계산: 정렬 기반 의도 키, 이웃 키 오류율, WPM 등 | `metrics.py` |
| 3 | 기술 통계 | `stats.describe` |
| 4 | 정규성 확인 | `stats.normality` |
| 5 | 2×2 반복측정 분산분석 + ART (H2) | `stats.rm_anova`, `stats.art_anova` |
| 6 | 사전 지정 비교 H1·H4, 효과 크기, 95% 신뢰구간 | `stats.planned_comparisons` |
| 7 | 민감도 분석: arcsine 변환, 키 입력 단위 GLMM | `stats.arcsine_sensitivity`, `stats.glmm` |
| 8 | 보조 분석: 오프라인 재생(섀도 판정), 순서 효과, 선호 | `stats.shadow_comparison` 등 |
| 9 | 그래프와 결과 저장 | `plots.py` |

분석 방법은 실험계획서(A8, 2026-11-03)에서 데이터 수집 전에 정한 것을 따른다. 각 함수의 설명(왜·어떻게)은 `.py` 파일의 주석에 있다.

**준비물**
- 익명화 원자료 `data/raw/S01.jsonl … S12.jsonl`
- Python 3 + numpy, scipy, pandas, matplotlib, statsmodels

> ⚠️ **커밋 전에 출력 지우기.** 실행 결과에는 참가자 데이터가 들어 있다. 이 노트북을 저장소에 다시 올릴 때는 *Edit → Clear All Outputs* 를 한 뒤 저장한다.

In [ ]:
import os, sys, warnings
from pathlib import Path
import pandas as pd

# 노트북을 analysis/ 에서 열든 저장소 맨 위에서 열든 동작하도록 경로를 맞춘다
ROOT = Path.cwd() if (Path.cwd() / "analysis").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "analysis"))
DATA = ROOT / "data" / "raw"          # 익명화 원자료 폴더 (다른 곳에 두었으면 여기만 바꾼다)
OUT = ROOT / "analysis_out"           # 표·그래프 저장 폴더 (참가자 데이터라 git 제외)
OUT.mkdir(exist_ok=True)

import parse_logs, metrics, stats, plots
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=UserWarning)
print("원자료 폴더:", DATA)

## 1. 원자료 불러오기

참가자 한 명 = 파일 하나(`S01.jsonl`). 파일 한 줄이 이벤트 하나다(터치, 문장 끝, 설문 등).
앱이 붙인 참가자 번호(P01–P04)는 순서 그룹마다 겹치므로, 참가자는 파일 이름(S01–S12)으로 구분한다.

In [ ]:
sessions = parse_logs.load_all(DATA)
participants = parse_logs.participants_table(sessions)
participants

In [ ]:
# 이벤트 종류별 개수: 어떤 기록이 얼마나 있는지 확인
events = parse_logs.events_frame(sessions)
events.groupby(["participant", "ev"]).size().unstack(fill_value=0)

In [ ]:
# 본 과제 터치 기록 예시: 같은 터치를 여러 판정으로 계산한 값(keyBase = 기본, keyCal = 보정, keyShown = 화면에 보인 키)
touches = parse_logs.events_frame(sessions, kinds=["touch"])
touches[touches.phase == "main"][["participant", "cond", "trial", "ux", "uy", "keyBase", "keyCal", "keyShown"]].head(10)

## 2. 지표 계산

조건마다 8문장의 키 입력을 모아 지표를 계산한다. 핵심은 **의도 키**(참가자가 누르려던 키)를 정하는 방법이다.

- 목표 문장과 최종 입력을 최소 편집 거리로 정렬해, 각 키 입력이 목표 문장의 어느 글자를 노린 것인지 찾는다.
- 단순히 "커서 위치의 글자"로 보면, 장갑 터치가 한 번 인식되지 않을 때 그 뒤가 전부 오류로 세진다.
- 이웃 키 오류는 의도 키 대신 그 **옆 키**(키 사이 거리 0.5키 폭 이하)가 입력된 경우다.

아래는 문장 하나를 예로 들어 의도 키가 어떻게 붙는지 보여 준다.

In [ ]:
# 예시: 첫 참가자, 조건 B(기본·장갑)의 첫 문장
trials, _ = metrics.collect_trials(sessions["S01"])
end, strokes = trials["B"][0]
aligned, omissions, insertions, n_chars, typed = metrics.align_trial(end["phrase"], strokes)
print("목표 문장:", end["phrase"])
print("최종 입력:", typed)
example = pd.DataFrame(aligned)[["intended", "decoded", "base", "cal"]]
example["오류"] = (example.decoded != "backspace") & example.intended.notna() & (example.decoded != example.intended)
example["이웃 키 오류"] = [bool(o and metrics.is_neighbor(i, d)) for o, i, d in zip(example["오류"], example.intended, example.decoded)]
example

In [ ]:
# 전체 계산: 참가자 × 조건(48줄), 문장(384줄), 키 입력(백스페이스 제외) 세 가지 표
per_condition, per_trial, keystrokes = metrics.build_tables(sessions)
print("참가자 × 조건:", per_condition.shape, "| 문장:", per_trial.shape, "| 키 입력:", keystrokes.shape)

# 데이터 점검: 조건당 8문장, 재시작 0, 시뮬레이션 없음, 입력 재생 결과 = 기록된 입력
checks = {
    "조건당 문장 수": sorted(per_condition.trials.unique()),
    "재시작 수": int(per_condition.restarts.sum()),
    "시뮬레이션 터치": bool(per_condition.simulated.any()),
    "재생 입력 ≠ 기록 입력": int((~per_trial.typed_matches_log).sum()),
}
checks

In [ ]:
per_condition[["participant", "group", "order", "cond", "position", "keystrokes", "neighbor_errors",
               "neighbor_error_rate", "key_error_rate", "wpm", "backspaces", "tlx_raw", "discomfort", "sus"]]

## 3. 기술 통계

검정 전에 조건별 평균·표준편차·중앙값을 본다. 평균과 중앙값이 크게 다르면 분포가 치우쳤다는 신호다.

**지표의 역할** (계획서 A8)
- 주 지표: **이웃 키 오류율**. 확증 결론(H1·H2)은 이 지표로만 내린다.
- 부 지표: 나머지 지표. 같은 틀로 분석하되 **탐색적**이라고 표시한다.

In [ ]:
METRICS = [  # (열 이름, 표시 이름, 배율, 낮을수록 좋은가, 서열 척도인가)
    ("neighbor_error_rate", "이웃 키 오류율 (%) [주]", 100, True, False),
    ("key_error_rate", "키 오류율 (%)", 100, True, False),
    ("wpm", "WPM", 1, False, False),
    ("backspaces", "백스페이스 수", 1, True, False),
    ("msd_rate", "최종 오류율 MSD (%)", 100, True, False),
    ("omission_rate", "누락률 (%)", 100, True, False),
    ("tlx_raw", "NASA-TLX", 1, True, False),
    ("discomfort", "불편도 (1–7)", 1, True, True),
    ("sus", "SUS", 1, False, False),
]
stats.describe(per_condition, [(c, l, s) for c, l, s, *_ in METRICS])

## 4. 정규성 확인 (Shapiro-Wilk)

대응 t검정과 반복측정 분산분석은 사람별 차이가 정규분포를 따른다고 가정한다.
p < .05면 정규성이 의심되므로 비모수 결과(Wilcoxon·ART)를 더 믿는다. 계획서에서 "정규성이 의심되면 변환이나 비모수 대안을 쓴다"고 정했다.

In [ ]:
stats.normality(per_condition, [(c, l, s) for c, l, s, *_ in METRICS])

## 5. 2×2 반복측정 분산분석 + ART 분산분석

- **반복측정 분산분석 (모수):** 키보드 효과, 손 상태 효과, **상호작용(H2: 보정의 이득이 장갑에서 더 큰가)** 을 검정한다.
- **ART 분산분석 (비모수):** 값을 순위로 바꿔 같은 세 효과를 검정한다. 정규분포 가정이 필요 없다.

두 결과가 같으면 분포 가정에 흔들리지 않는 결론이다. 부분 η²는 그 효과가 설명하는 변동의 비율(0–1)이다.

In [ ]:
# 주 지표
pd.concat([stats.rm_anova(per_condition, "neighbor_error_rate"),
           stats.art_anova(per_condition, "neighbor_error_rate")[["ART F", "p"]].add_prefix("ART ")], axis=1)

In [ ]:
# 모든 지표 (주 지표 외에는 탐색적)
anova_all = pd.concat({label: pd.concat([stats.rm_anova(per_condition, col).set_index("효과"),
                                          stats.art_anova(per_condition, col).set_index("효과")[["ART F", "p"]].add_prefix("ART ")], axis=1)
                       for col, label, *_ in METRICS})
anova_all

## 6. 사전 지정 비교: H1 (장갑 B→D), H4 (맨손 A→C)

차이 = 보정 − 기본. 오류·부담 지표는 음수가, WPM·SUS는 양수가 좋은 쪽이다.

- **t검정 / Wilcoxon:** 같은 질문을 평균과 순위로 각각 검정한다. 두 비교의 p값에는 Holm 보정을 한다.
- **95% 신뢰구간:** 차이가 얼마나 큰지 보여 준다. p값은 차이가 있는지만 알려 준다.
- **dz:** 사람마다 얼마나 일정하게 차이가 났는지까지 반영한 효과 크기다. 0.2 작음, 0.5 중간, 0.8 이상 큼.
- **불편도:** 서열 척도라 Wilcoxon을 주 결과로 본다.
- **H4 해석:** p가 큰 것은 "차이를 찾지 못했다"는 뜻이지 "차이가 없다"는 증명이 아니다. 신뢰구간으로 "차이가 있더라도 이 정도 이하"라고 보고한다.

In [ ]:
planned = {label: stats.planned_comparisons(per_condition, col, scale, lower, ordinal)
           for col, label, scale, lower, ordinal in METRICS}
planned_all = pd.concat(planned)
planned_all

In [ ]:
# 장갑 손해 감소율: 장갑 때문에 나빠진 만큼(장갑 − 맨손) 중 보정 키보드가 몇 %를 줄였나
pd.DataFrame({label: stats.glove_penalty(per_condition, col, scale)
              for col, label, scale, *_ in METRICS[:3]}).T

## 7. 민감도 분석

### 7-1. arcsine-sqrt 변환
비율 데이터를 변환한 뒤 같은 분산분석을 다시 해서, 결론이 같은지 확인한다.

In [ ]:
stats.arcsine_sensitivity(per_condition, "neighbor_error_rate")

### 7-2. 키 입력 단위 로지스틱 혼합모형 (GLMM)

주 분석은 참가자 × 조건 **평균 비율**을 쓴다. GLMM은 키 입력 하나하나(약 1만 3천 개)가 이웃 키 오류였는지(0/1)를 직접 모형화하고, 참가자 차이와 문장 차이를 무작위 효과로 따로 떼어 낸다.

- **모형:** `logit P(이웃 키 오류) = b0 + b1·보정 + b2·장갑 + b3·보정×장갑 + (1|참가자) + (1|문장)`
- **b3 < 0:** 보정의 효과가 장갑에서 더 크다(H2).
- **오즈비 = exp(계수):** 1보다 작으면 오류 가능성이 줄어든다.
- **적합 방법 두 가지:**
  - **베이즈 GLMM:** 참가자·문장 무작위 효과를 모두 넣는다. 사후 표준편차가 좁게 나오는 경향이 있어 방향·크기 확인용으로 쓴다.
  - **GEE:** 참가자 군집 강건 표준오차로 p값과 신뢰구간을 낸다.

In [ ]:
g = stats.glmm(keystrokes)
print("키 입력 수:", g["n_keystrokes"])
display(g["bayes_fixed"], g["bayes_random_sd"])

In [ ]:
display(g["gee"], g["gee_simple_effects"])

## 8. 보조 분석

### 8-1. 오프라인 재생 (섀도 판정)
앱은 모든 터치를 기본·보정(M0, M1) 판정으로 동시에 계산해 기록했다. 같은 터치를 판정만 바꿔 다시 읽으면 **판정 방식 자체의 효과**를 볼 수 있다. 다만 참가자는 화면에 보인 판정을 보고 고치며 입력했으므로, 이 결과는 보조 증거로만 쓴다.

In [ ]:
stats.shadow_comparison(per_condition)

### 8-2. 순서 효과 확인과 선호
라틴 방격이 순서 효과(뒤에 할수록 연습돼 유리해지는 것)를 상쇄했는지 확인한다. 6명 대 6명이라 검정력이 낮으므로, "순서와 관계없이 같은 방향"인지만 본다.

In [ ]:
by_position, order_tab = stats.order_check(per_condition)
display(by_position, order_tab)

In [ ]:
stats.preferences(per_condition)

## 9. 그래프와 결과 저장

그래프는 `analysis_out/`에 PNG로 저장되며, 슬라이드에 그대로 쓸 수 있다.

In [ ]:
plots.use_korean_font()
FIG = [("neighbor_error_rate", "이웃 키 오류율 (%)", 100), ("wpm", "WPM", 1), ("tlx_raw", "NASA-TLX", 1), ("sus", "SUS", 1)]
fig1 = plots.interaction(per_condition, FIG)
fig1.savefig(OUT / "fig1_interaction.png", dpi=200, bbox_inches="tight")

In [ ]:
fig2 = plots.participants(per_condition)
fig2.savefig(OUT / "fig2_participants.png", dpi=200, bbox_inches="tight")

In [ ]:
fig3 = plots.differences({
    "이웃 키 오류율 (%p)": planned["이웃 키 오류율 (%) [주]"],
    "WPM": planned["WPM"], "NASA-TLX": planned["NASA-TLX"], "SUS": planned["SUS"],
})
fig3.savefig(OUT / "fig3_differences.png", dpi=200, bbox_inches="tight")

In [ ]:
# 표 저장 (보고서·발표 자료용)
per_condition.to_csv(OUT / "per_condition.csv", index=False)
per_trial.to_csv(OUT / "per_trial.csv", index=False)
anova_all.to_csv(OUT / "anova.csv")
planned_all.to_csv(OUT / "planned_comparisons.csv")
g["gee"].to_csv(OUT / "glmm_gee.csv", index=False)
sorted(p.name for p in OUT.iterdir())